# RoBERTa-base - Fake Job Detection
**Model:** `roberta-base` (Facebook AI)

**Why RoBERTa over MiniLM/DeBERTa for this task:**
- Trained on 10x more data than BERT with dynamic masking
- Stable fp16 training - no nan/zero-loss issues like DeBERTa-v3
- Consistently outperforms BERT-family models on text classification benchmarks
- Handles long job descriptions well with 512 token limit
- Well-supported by captum for Integrated Gradients explainability

In [1]:
FAST_MODE = True

## Install Dependencies

In [2]:
!pip install captum datasets evaluate transformers[torch] accelerate -q

## Imports

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from captum.attr import IntegratedGradients
import warnings
warnings.filterwarnings('ignore')

import transformers, accelerate
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"torch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

transformers: 5.3.0
accelerate:   1.13.0
torch:        2.11.0+cu126
CUDA available: True


## Load Data

In [ ]:
df = pd.read_csv("fake_job_postings_augmented.csv")

print(f"Dataset loaded: {df.shape}")
print(f"Fraud rate: {df['fraudulent'].mean()*100:.2f}%")
print(df['fraudulent'].value_counts())

Dataset loaded: (29880, 18)
Fraud rate: 43.06%
fraudulent
0    17014
1    12866
Name: count, dtype: int64


## Combine Text Fields

In [ ]:
def combine_text_fields(row):
    fields = [
        'title', 'location', 'company_profile', 'description',
        'requirements', 'benefits', 'required_experience',
        'required_education', 'industry', 'function'
    ]
    text_parts = []
    for field in fields:
        if pd.notna(row[field]):
            text_parts.append(str(row[field]))
    return ' '.join(text_parts) if text_parts else "unknown job"

print("Combining text fields...")
df['combined_text'] = df.apply(combine_text_fields, axis=1)

df = df[['combined_text', 'fraudulent']].dropna()
df['combined_text'] = df['combined_text'].astype(str)
df['fraudulent']    = df['fraudulent'].astype(int)

print(f"After dropna: {df.shape}")
print(f"Fraud rate:   {df['fraudulent'].mean()*100:.2f}%")

Combining text fields...
After dropna: (29880, 2)
Fraud rate:   43.06%


## Train / Val / Test Split (80 / 10 / 10)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['fraudulent']
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.0f}%)")
print(f"\nFraud rates — Train: {train_df['fraudulent'].mean()*100:.1f}%  "
      f"Val: {val_df['fraudulent'].mean()*100:.1f}%  "
      f"Test: {test_df['fraudulent'].mean()*100:.1f}%")

Train: 23904 (80%)
Val:   2988 (10%)
Test:  2988 (10%)

Fraud rates — Train: 43.1%  Val: 43.1%  Test: 43.0%


## Load RoBERTa-base

In [7]:
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "legitimate", 1: "fraud"},
    label2id={"legitimate": 0, "fraud": 1}
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model:    {MODEL_NAME}")
print(f"Device:   {device}")
print(f"Class 0:  {model.config.id2label[0]}")
print(f"Class 1:  {model.config.id2label[1]}")
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Params:   {n_params:.1f}M")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model:    roberta-base
Device:   cuda
Class 0:  legitimate
Class 1:  fraud
Params:   124.6M


## Tokenize Datasets

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["combined_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)

train_ds = train_ds.rename_column("fraudulent", "labels")
val_ds   = val_ds.rename_column("fraudulent", "labels")
test_ds  = test_ds.rename_column("fraudulent", "labels")

# RoBERTa does NOT use token_type_ids
for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples")
print(f"Test:  {len(test_ds)} samples")

Map:   0%|          | 0/23904 [00:00<?, ? examples/s]

Map:   0%|          | 0/2988 [00:00<?, ? examples/s]

Map:   0%|          | 0/2988 [00:00<?, ? examples/s]

Train: 23904 samples
Val:   2988 samples
Test:  2988 samples


## Fine-tune RoBERTa

In [ ]:
training_args = TrainingArguments(
    output_dir='./roberta_results',
    num_train_epochs=1 if FAST_MODE else 3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),   
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("Starting RoBERTa training...")
trainer.train()
print("\nTraining complete!")

Starting RoBERTa training...


Epoch,Training Loss,Validation Loss
1,0.048070,0.035538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training complete!


## Evaluate on Test Set

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

model.eval()
fraud_probs = []
true_labels = []

with torch.no_grad():
    for batch in test_ds:
        inputs = {
            'input_ids':      batch['input_ids'].unsqueeze(0).to(device),
            'attention_mask': batch['attention_mask'].unsqueeze(0).to(device),
        }
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        fraud_probs.append(probs[0][1].item())
        true_labels.append(batch['labels'].item())

best_f1 = 0
best_threshold = 0.5

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = [1 if p > threshold else 0 for p in fraud_probs]
    f1 = f1_score(true_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Best threshold: {best_threshold}")
predictions = [1 if p > best_threshold else 0 for p in fraud_probs]

print("\nClassification Report:")
print(classification_report(true_labels, predictions,
                            target_names=['Legitimate', 'Fraud'], digits=3))

cm = confusion_matrix(true_labels, predictions)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix:")
print(f"                Predicted")
print(f"                Legit  Fraud")
print(f"Actual Legit    {tn:4d}  {fp:4d}")
print(f"       Fraud    {fn:4d}  {tp:4d}")
print(f"\nFraud Recall:    {tp/(tp+fn)*100:.1f}%")
print(f"Precision:       {tp/(tp+fp)*100:.1f}%")
print(f"F1-Score:        {best_f1:.3f}")

Best threshold: 0.3

Classification Report:
              precision    recall  f1-score   support

  Legitimate      0.985     0.996     0.991      1702
       Fraud      0.995     0.981     0.988      1286

    accuracy                          0.990      2988
   macro avg      0.990     0.989     0.989      2988
weighted avg      0.990     0.990     0.990      2988

Confusion Matrix:
                Predicted
                Legit  Fraud
Actual Legit    1696     6
       Fraud      25  1261

Fraud Recall:    98.1%
Precision:       99.5%
F1-Score:        0.988


## Save Model

In [11]:
from pathlib import Path

save_dir = Path('../models/model_roberta_final')
save_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ..\models\model_roberta_final


## Integrated Gradients — Explainability

RoBERTa uses SentencePiece-style tokenization. Special tokens are `<s>` (CLS) and `</s>` (SEP).
Subword tokens starting with `Ġ` are word-initial pieces — we clean these in the aggregate analysis.

In [ ]:
def forward_pass(input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return F.softmax(outputs.logits, dim=1)[:, 1]

def forward_pass_embeds(inputs_embeds, attention_mask):
    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    return F.softmax(outputs.logits, dim=1)[:, 1]

ig = IntegratedGradients(forward_pass_embeds)

word_embeddings = model.base_model.embeddings.word_embeddings

def get_attributions(text, n_steps=50):
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length'
    ).to(device)

    inputs_embeds   = word_embeddings(encoded['input_ids']).detach().requires_grad_(True)
    baseline_embeds = torch.zeros_like(inputs_embeds).to(device)

    attributions, delta = ig.attribute(
        inputs=inputs_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(encoded['attention_mask'],),
        return_convergence_delta=True,
        n_steps=n_steps
    )

    attributions_sum  = attributions.sum(dim=-1).squeeze(0)
    attributions_norm = attributions_sum / torch.norm(attributions_sum)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

    with torch.no_grad():
        fraud_prob = forward_pass(encoded['input_ids'], encoded['attention_mask']).item()

    return tokens, attributions_norm.detach().cpu().numpy(), fraud_prob, delta.item()


def visualize_attributions(text, attributions, tokens, top_k=15):
    special = ['<s>', '</s>', '<pad>']
    filtered = [(tok, attr) for tok, attr in zip(tokens, attributions)
                if tok not in special]
    sorted_attr = sorted(filtered, key=lambda x: x[1], reverse=True)

    print("\n" + "="*80)
    print("WORD ATTRIBUTION ANALYSIS")
    print("="*80)
    print(f"\nText: {text[:150]}...")
    print(f"\nTop {top_k} Fraud Indicators (positive attribution):")
    for i, (token, attr) in enumerate(sorted_attr[:top_k], 1):
        # Clean Ġ prefix for display (RoBERTa word-initial marker)
        display_tok = token.replace('Ġ', '')
        print(f"  {i:2d}. {display_tok:20s} -> {attr:+.4f}")
    print(f"\nTop 5 Legitimacy Indicators (negative attribution):")
    for i, (token, attr) in enumerate(sorted_attr[-5:], 1):
        display_tok = token.replace('Ġ', '')
        print(f"  {i:2d}. {display_tok:20s} -> {attr:+.4f}")

print("IG setup complete.")

IG setup complete.


## Example 1: High-Confidence Fraud Detection

In [13]:
print("="*80)
print("EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION")
print("="*80)

fraud_examples = test_df[test_df['fraudulent'] == 1]
text = fraud_examples.iloc[0]['combined_text']

tokens, attributions, fraud_prob, delta = get_attributions(text)

print(f"\nModel Prediction: {fraud_prob:.1%} fraud probability")
print(f"Actual Label: FRAUD")
print(f"{'Correct!' if fraud_prob > best_threshold else 'Missed!'}")

visualize_attributions(text, attributions, tokens, top_k=15)
print(f"\nConvergence Delta: {delta:.6f} (closer to 0 is better)")

EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION

Model Prediction: 100.0% fraud probability
Actual Label: FRAUD
Correct!

WORD ATTRIBUTION ANALYSIS

Text: Home Logistics Coordinator GH, AA, Accra Rapid Enterprises connects international shoppers with domestic shipping agents, ensuring secure and timely d...

Top 15 Fraud Indicators (positive attribution):
   1. $                    -> +0.3154
   2. .                    -> +0.2510
   3. .                    -> +0.2316
   4. 75                   -> +0.2212
   5. Company              -> +0.2002
   6. .                    -> +0.1670
   7. .                    -> +0.1574
   8. shipping             -> +0.1548
   9. home                 -> +0.1485
  10. .                    -> +0.1480
  11. shipping             -> +0.1424
  12. .                    -> +0.1422
  13. earnings             -> +0.1383
  14. .                    -> +0.1364
  15. All                  -> +0.1285

Top 5 Legitimacy Indicators (negative attribution):
   1. our               

## Example 2: High-Confidence Legitimate Detection

In [14]:
print("="*80)
print("EXAMPLE 2: HIGH-CONFIDENCE LEGITIMATE DETECTION")
print("="*80)

legit_examples = test_df[test_df['fraudulent'] == 0]
text_legit = legit_examples.iloc[0]['combined_text']

tokens_l, attributions_l, fraud_prob_l, delta_l = get_attributions(text_legit)

print(f"\nModel Prediction: {fraud_prob_l:.1%} fraud probability")
print(f"Actual Label: LEGITIMATE")
print(f"{'Correct!' if fraud_prob_l < best_threshold else 'False positive!'}")

visualize_attributions(text_legit, attributions_l, tokens_l, top_k=15)
print(f"\nConvergence Delta: {delta_l:.6f}")

EXAMPLE 2: HIGH-CONFIDENCE LEGITIMATE DETECTION

Model Prediction: 0.0% fraud probability
Actual Label: LEGITIMATE
Correct!

WORD ATTRIBUTION ANALYSIS

Text: Senior SQL & Database Developer GB, LND, Shoreditch Adthena is the UK’s leading competitive intelligence service for Google search advertisers. Adthen...

Top 15 Fraud Indicators (positive attribution):
   1. SQL                  -> +0.4804
   2. a                    -> +0.2178
   3. Senior               -> +0.1269
   4. then                 -> +0.1065
   5. Sh                   -> +0.0986
   6. Ad                   -> +0.0914
   7. ND                   -> +0.0902
   8. then                 -> +0.0861
   9. major                -> +0.0823
  10. .                    -> +0.0807
  11. Ad                   -> +0.0748
  12. ef                   -> +0.0742
  13. ored                 -> +0.0612
  14. .                    -> +0.0501
  15. afe                  -> +0.0478

Top 5 Legitimacy Indicators (negative attribution):
   1. UK        

## Example 3: False Negatives (Missed Frauds)

In [15]:
print("="*80)
print("EXAMPLE 3: FALSE NEGATIVE (MISSED FRAUD)")
print("="*80)

missed_frauds_idx = [
    i for i, (true, pred) in enumerate(zip(true_labels, predictions))
    if true == 1 and pred == 0
]

if missed_frauds_idx:
    fn_example = test_df.iloc[missed_frauds_idx[0]]
    text_fn = fn_example['combined_text']

    tokens_fn, attributions_fn, fraud_prob_fn, delta_fn = get_attributions(text_fn)

    print(f"\nModel Prediction: {fraud_prob_fn:.1%} fraud probability (below {best_threshold} threshold)")
    print(f"Actual Label: FRAUD")
    print(f"MISSED - Why did the model fail?")

    visualize_attributions(text_fn, attributions_fn, tokens_fn, top_k=15)
    print(f"\nConvergence Delta: {delta_fn:.6f}")
    print("\nAnalysis: This fraud likely lacks obvious fraud keywords and appears more professional.")
else:
    print("\nNo false negatives found in test set!")

EXAMPLE 3: FALSE NEGATIVE (MISSED FRAUD)

Model Prediction: 3.4% fraud probability (below 0.3 threshold)
Actual Label: FRAUD
MISSED - Why did the model fail?

WORD ATTRIBUTION ANALYSIS

Text: customer service representative US, CA, san jose  What we're looking for:Do you like helping others? Are you a customer service pro? If you answered y...

Top 15 Fraud Indicators (positive attribution):
   1. R                    -> +0.4612
   2. a                    -> +0.2854
   3. Rep                  -> +0.1882
   4. to                   -> +0.1473
   5. US                   -> +0.1251
   6. customer             -> +0.1234
   7. email                -> +0.1202
   8. pro                  -> +0.1183
   9. .                    -> +0.1065
  10. Customer             -> +0.1034
  11. representative       -> +0.0979
  12. ,                    -> +0.0949
  13. .                    -> +0.0859
  14. support              -> +0.0843
  15. next                 -> +0.0819

Top 5 Legitimacy Indicators (negat

## Aggregate: Top Fraud Indicator Words

In [ ]:
print("="*80)
print("AGGREGATE ANALYSIS: TOP FRAUD INDICATORS ACROSS TEST SET")
print("="*80)

word_attributions = {}
fraud_samples = test_df[test_df['fraudulent'] == 1].head(50)
print(f"\nAnalyzing {len(fraud_samples)} fraud examples...")

for idx, row in fraud_samples.iterrows():
    tokens_a, attrs_a, _, _ = get_attributions(row['combined_text'], n_steps=25)

    special = ['<s>', '</s>', '<pad>']
    for token, attr in zip(tokens_a, attrs_a):
        clean_tok = token.replace('Ġ', '')
        if token not in special and len(clean_tok) > 1:
            word_attributions.setdefault(clean_tok, []).append(attr)

mean_attributions = {
    word: np.mean(attrs)
    for word, attrs in word_attributions.items()
    if len(attrs) >= 3
}

sorted_words = sorted(mean_attributions.items(), key=lambda x: x[1], reverse=True)

print("\nTOP 20 FRAUD INDICATOR WORDS (across all fraud examples):")
for i, (word, attr) in enumerate(sorted_words[:20], 1):
    count = len(word_attributions[word])
    print(f"  {i:2d}. {word:20s} -> {attr:+.4f} (appeared in {count} examples)")

print("\nIntegrated Gradients analysis complete!")

AGGREGATE ANALYSIS: TOP FRAUD INDICATORS ACROSS TEST SET

Analyzing 50 fraud examples...

TOP 20 FRAUD INDICATOR WORDS (across all fraud examples):
   1. TX                   -> +0.3556 (appeared in 3 examples)
   2. oil                  -> +0.3248 (appeared in 3 examples)
   3. No                   -> +0.1917 (appeared in 43 examples)
   4. 150                  -> +0.1628 (appeared in 3 examples)
   5. Complete             -> +0.1320 (appeared in 5 examples)
   6. shipping             -> +0.1309 (appeared in 3 examples)
   7. earnings             -> +0.1255 (appeared in 5 examples)
   8. earn                 -> +0.1204 (appeared in 15 examples)
   9. 59                   -> +0.1195 (appeared in 3 examples)
  10. flights              -> +0.1164 (appeared in 3 examples)
  11. salary               -> +0.1148 (appeared in 3 examples)
  12. 200                  -> +0.1147 (appeared in 4 examples)
  13. Account              -> +0.1146 (appeared in 4 examples)
  14. Security             -> +

: 